In [2]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

In [3]:
orders = pd.read_csv("../data/orders(源数据).csv")
print(orders.columns)

Index(['order_id', 'customer_id', 'order_date', 'year', 'month', 'quarter',
       'is_weekend', 'day_of_week', 'product_name', 'category',
       'unit_price_usd', 'quantity', 'subtotal_usd', 'discount_pct',
       'discount_amount_usd', 'shipping_fee_usd', 'tax_pct', 'tax_amount_usd',
       'total_amount_usd', 'payment_method', 'device_used', 'delivery_days',
       'delivery_date', 'order_status', 'returned', 'customer_rating',
       'session_duration_minutes', 'pages_viewed_before_purchase',
       'is_repeat_customer'],
      dtype='object')


In [26]:
#对数据进行初步筛选、格式调整、排序
orders = orders[orders['order_status']=='Delivered'].copy()
orders['order_date'] = pd.to_datetime(orders['order_date'])
orders = orders.sort_values(['customer_id','order_date'])

In [27]:
#提取每个用户首单
first_orders = orders.groupby ('customer_id').first().reset_index()
#提取用户第二单
multi_order_users = (orders.groupby('customer_id').filter(lambda x: len(x) >= 2))
second_order = (multi_order_users.groupby('customer_id').nth(1).reset_index())
second_order = second_order[ ['customer_id', 'order_date']]
second_order = second_order.rename(columns={'order_date': 'second_order_date'})
#将用户首单和第二单合并，计算90天内复购率
first_orders = first_orders.merge(second_order,on= 'customer_id',how = 'left')
first_orders['days_to_second_order'] = ( first_orders['second_order_date'] - first_orders['order_date'] ).dt.days
first_orders ['repurchase_90d'] = np.where((first_orders['days_to_second_order'].notna() ) &
   (first_orders['days_to_second_order'] <= 90), 1,0)

In [28]:
#检验样本情况——发现因变量不平衡）
print(first_orders['repurchase_90d'].value_counts(normalize=True))

repurchase_90d
0    0.897852
1    0.102148
Name: proportion, dtype: float64


In [29]:
#数据量足够，可以分训练集和测试集
train_df, test_df = train_test_split(
    first_orders,
    test_size=0.3,
    random_state=42)

In [31]:
# 针对因变量不平衡问题：用 class_weight 的思想，在 smf 中加权重
# 计算样本权重：让少数类（复购=1）获得更高权重
scale_pos = len(first_orders) / (2 * first_orders['repurchase_90d'].sum())  # 约 4.55
scale_neg = len(first_orders) / (2 * (len(first_orders) - first_orders['repurchase_90d'].sum()))  # 约 0.56
first_orders['sample_weight'] = first_orders['repurchase_90d'].map({1: scale_pos, 0: scale_neg})

model_weighted = smf.logit(
    "repurchase_90d ~ discount_pct + customer_rating + total_amount_usd + session_duration_minutes + pages_viewed_before_purchase + C(category)", 
    data=first_orders, 
    weights=first_orders['sample_weight']
).fit()
print(model_weighted.summary())
# 现在 p 值会比不加权时更准确

Optimization terminated successfully.
         Current function value: 0.363857
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:         repurchase_90d   No. Observations:                 5523
Model:                          Logit   Df Residuals:                     5504
Method:                           MLE   Df Model:                           18
Date:                Thu, 14 May 2026   Pseudo R-squ.:                0.003689
Time:                        21:48:32   Log-Likelihood:                -2009.6
converged:                       True   LL-Null:                       -2017.0
Covariance Type:            nonrobust   LLR p-value:                    0.6701
                                            coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------
Intercept                                -1.6692      

C:\python3\lib\site-packages\statsmodels\base\model.py:130: ValueWarning: unknown kwargs ['weights']
  warnings.warn(msg, ValueWarning)
C:\python3\lib\site-packages\statsmodels\base\model.py:130: ValueWarning: unknown kwargs ['weights']
  warnings.warn(msg, ValueWarning)


In [32]:
test_df['pred_prob'] = model_weighted.predict(test_df)
test_df = test_df.dropna(subset=['pred_prob'])
auc = roc_auc_score(
    test_df['repurchase_90d'],
    test_df['pred_prob'])
print("\nAUC:", auc)


AUC: 0.5509683159579782


In [33]:
coef_df = pd.DataFrame({
    'feature': model_weighted.params.index,
    'coef': model_weighted.params.values,
    'odds_ratio': np.exp(model_weighted.params.values),
    'p_value': model_weighted.pvalues.values})
print("\nCoefficient Table:")
print(coef_df)


Coefficient Table:
                                  feature      coef  odds_ratio   p_value
0                               Intercept -1.669163    0.188405  0.000009
1   C(category)[T.Beauty & Personal Care] -0.111088    0.894860  0.684194
2                    C(category)[T.Books] -0.008165    0.991868  0.975664
3       C(category)[T.Clothing & Apparel] -0.340290    0.711564  0.174135
4              C(category)[T.Electronics] -0.236145    0.789666  0.343685
5           C(category)[T.Food & Grocery] -0.048006    0.953128  0.862471
6        C(category)[T.Health & Wellness] -0.020405    0.979802  0.943288
7           C(category)[T.Home & Kitchen] -0.275604    0.759113  0.284845
8    C(category)[T.Jewelry & Accessories]  0.067345    1.069664  0.824670
9          C(category)[T.Office Supplies] -0.297456    0.742705  0.376016
10            C(category)[T.Pet Supplies]  0.114706    1.121544  0.714171
11       C(category)[T.Sports & Outdoors]  0.039173    1.039950  0.883663
12            C(ca